In [ ]:
# Impor modul untuk akses Google Drive
from google.colab import drive

# Hubungkan Google Drive ke Colab
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
import tensorflow as tf

dataset_dir = "/content/drive/MyDrive/FNNPK"
image_size = (224, 224)

def decode_image(file_path):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    return img

def load_images_from_folder(dataset_path):
    image_list, label_list = [], []
    class_names = sorted(os.listdir(dataset_path))

    for label in class_names:
        class_folder = os.path.join(dataset_path, label)
        if not os.path.isdir(class_folder):
            continue

        for file_name in os.listdir(class_folder):
            file_path = os.path.join(class_folder, file_name)
            try:
                img = decode_image(file_path)
                img = tf.image.resize(img, image_size)
                img = tf.cast(img, tf.float32) / 255.0
                image_list.append(img)
                label_list.append(label)
            except Exception as e:
                print(f"Skipping file {file_path}: {e}")

    return image_list, label_list

# Load data
print(f"⏳ Memuat data dari {dataset_dir}...")
images, labels = load_images_from_folder(dataset_dir)
print(f"✅ Data berhasil dimuat ({len(images)} gambar).")
print("Contoh label:", labels[:10])

In [ ]:
import numpy as np
from collections import Counter

# Mapping label <-> angka
label2id = {label: i for i, label in enumerate(sorted(set(labels)))}
id2label = {i: label for label, i in label2id.items()}
int_labels = [label2id[l] for l in labels]

# Konversi ke numpy array
X = np.array([img.numpy() for img in images])
y = np.array(int_labels)

print("🔢 Label2ID:", label2id)
print("📊 Distribusi awal:", Counter(y))
print("🖼️ X shape:", X.shape, "y shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter

# Step 1: Split train 60% dan sisanya 40%
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)

# Step 2: Split sisa 40% menjadi val dan test masing-masing 50% dari 40% → 20% dari total
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("📊 Distribusi train:", Counter(y_train))
print("📊 Distribusi val:", Counter(y_val))
print("📊 Distribusi test:", Counter(y_test))

In [ ]:
from PIL import Image

base_dir = "/content/drive/MyDrive/FNNPK_SPLIT"
for split, X_split, y_split in [("train", X_train, y_train),
                                ("val", X_val, y_val),
                                ("test", X_test, y_test)]:
    split_dir = os.path.join(base_dir, split)
    os.makedirs(split_dir, exist_ok=True)

    for idx, (img_array, label_id) in enumerate(zip(X_split, y_split)):
        label_name = id2label[label_id]
        label_folder = os.path.join(split_dir, label_name)
        os.makedirs(label_folder, exist_ok=True)

        img_pil = Image.fromarray((img_array * 255).astype(np.uint8))
        img_pil.save(os.path.join(label_folder, f"{split}_{idx}.jpg"))

print("✅ Dataset sudah disimpan ke folder FNNPK_SPLIT (train/val/test).")

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir + "/train", image_size=(224,224), batch_size=32
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir + "/val", image_size=(224,224), batch_size=32
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir + "/test", image_size=(224,224), batch_size=32
)

print(train_ds)

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

# Path dataset
train_path = "/content/drive/MyDrive/FNNPK_SPLIT/train"

# Load dataset train
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    labels="inferred",
    label_mode="categorical",
    image_size=(224, 224),
    shuffle=True,
    seed=42,
    batch_size=16
)

# Ambil satu batch
class_names = train_ds.class_names
images, labels = next(iter(train_ds))

plt.figure(figsize=(15, 15))
for i in range(15):
    ax = plt.subplot(5, 5, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i].numpy().argmax()])
    plt.axis("off")
plt.show()

In [ ]:
import os

# Path dataset hasil split
base_dir = "/content/drive/MyDrive/FNNPK_SPLIT"

def count_images_per_class(split_dir):
    split_path = os.path.join(base_dir, split_dir)
    class_counts = {}
    for class_name in sorted(os.listdir(split_path)):
        class_folder = os.path.join(split_path, class_name)
        if os.path.isdir(class_folder):
            files = [f for f in os.listdir(class_folder) if f.lower().endswith(('.jpg','.jpeg','.png'))]
            class_counts[class_name] = len(files)
    return class_counts

# Hitung distribusi
train_counts = count_images_per_class("train")
val_counts   = count_images_per_class("val")
test_counts  = count_images_per_class("test")

# Total semua
total_images = sum(train_counts.values()) + sum(val_counts.values()) + sum(test_counts.values())

print(f"📸 Total data keseluruhan: {total_images} gambar\n")

print("📊 Distribusi data setelah split:")

print("  🟢 Train:")
for cls, count in train_counts.items():
    print(f"    {cls}: {count} gambar")

print("  🟡 Validation:")
for cls, count in val_counts.items():
    print(f"    {cls}: {count} gambar")

print("  🔵 Test:")
for cls, count in test_counts.items():
    print(f"    {cls}: {count} gambar")

In [ ]:
import os
from collections import Counter
import numpy as np

# base path dataset hasil split
base_dir = "/content/drive/MyDrive/FNNPK_SPLIT"
train_dir = os.path.join(base_dir, "train")
val_dir   = os.path.join(base_dir, "val")
test_dir  = os.path.join(base_dir, "test")

# ambil daftar kelas (dari folder di train)
class_names = sorted(os.listdir(train_dir))
label2id = {c: i for i, c in enumerate(class_names)}
id2label = {i: c for c, i in label2id.items()}

print("🏷️ Mapping label:", label2id)

In [ ]:
def load_paths_and_labels(split_dir):
    paths, labels = [], []
    for cls in sorted(os.listdir(split_dir)):
        cls_path = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_path):
            continue
        for f in os.listdir(cls_path):
            if f.lower().endswith(('.jpg','.jpeg','.png')):
                paths.append(os.path.join(cls_path, f))
                labels.append(label2id[cls])
    return paths, labels

train_paths, train_labels = load_paths_and_labels(train_dir)
val_paths, val_labels     = load_paths_and_labels(val_dir)
test_paths, test_labels   = load_paths_and_labels(test_dir)

print("📊 Jumlah data per split:")
print("  Train:", len(train_paths))
print("  Val:", len(val_paths))
print("  Test:", len(test_paths))

print("🎯 Distribusi Train:", Counter(train_labels))
print("🎯 Distribusi Val:", Counter(val_labels))
print("🎯 Distribusi Test:", Counter(test_labels))

In [ ]:
import os
import shutil
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img, save_img

# path dataset hasil split
base_dir = "/content/drive/MyDrive/FNNPK_SPLIT"
train_dir = os.path.join(base_dir, "train")
val_dir   = os.path.join(base_dir, "val")
test_dir  = os.path.join(base_dir, "test")

IMG_SIZE = (224, 224)

In [ ]:
def oversample_class(train_dir, target_class, target_count):
    class_dir = os.path.join(train_dir, target_class)
    files = os.listdir(class_dir)
    current_count = len(files)

    if current_count >= target_count:
        print(f"✔ {target_class} sudah cukup ({current_count} gambar).")
        return

    i = 0
    while len(os.listdir(class_dir)) < target_count:
        src = os.path.join(class_dir, files[i % current_count])
        dst = os.path.join(class_dir, f"copy_{i}_{files[i % current_count]}")
        shutil.copy(src, dst)
        i += 1

    print(f"✅ Oversampling {target_class}: dari {current_count} → {len(os.listdir(class_dir))} gambar")

target_size = 50
for class_name in os.listdir(train_dir):
    oversample_class(train_dir, class_name, target_size)

In [ ]:
aug_datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

def augment_images(class_dir, augment_count=3):
    files = os.listdir(class_dir)
    for file in files:
        img_path = os.path.join(class_dir, file)
        img = load_img(img_path, target_size=IMG_SIZE)
        x = img_to_array(img)
        x = np.expand_dims(x, axis=0)

        # generate augment_count versi baru
        aug_iter = aug_datagen.flow(x, batch_size=1, save_to_dir=class_dir, save_prefix="aug", save_format="jpg")
        for i in range(augment_count):
            next(aug_iter)

# contoh augment semua kelas
for class_name in os.listdir(train_dir):
    print(f"🔄 Augmentasi kelas {class_name}...")
    augment_images(os.path.join(train_dir, class_name), augment_count=3)

In [ ]:
train_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode="categorical"
)

val_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode="categorical"
)

test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

In [ ]:
import os

# path dataset final
base_dir = "/content/drive/MyDrive/FNNPK_SPLIT"

def count_images(split_dir):
    split_path = os.path.join(base_dir, split_dir)
    class_counts = {}
    for cls in sorted(os.listdir(split_path)):
        class_folder = os.path.join(split_path, cls)
        if os.path.isdir(class_folder):
            files = [f for f in os.listdir(class_folder) if f.lower().endswith(('.jpg','.jpeg','.png'))]
            class_counts[cls] = len(files)
    return class_counts

train_counts = count_images("train")
val_counts   = count_images("val")
test_counts  = count_images("test")

print("📊 Distribusi data:")
print("  🟢 Train:", train_counts)
print("  🟡 Validation:", val_counts)
print("  🔵 Test:", test_counts)

print(f"\n📸 Total gambar: {sum(train_counts.values()) + sum(val_counts.values()) + sum(test_counts.values())}")

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB1
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# ====== Setup folder dataset ======
base_dir = "/content/drive/MyDrive/FNNPK_SPLIT"
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# ====== Data Generator ======
train_datagen = ImageDataGenerator(
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1
)

val_test_datagen = ImageDataGenerator()

train_gen = train_datagen.flow_from_directory(
    os.path.join(base_dir, "train"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_gen = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, "val"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_gen = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, "test"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

# ====== Model EfficientNetB1 ======
input_layer = Input(shape=(224, 224, 3))
effnet = EfficientNetB1(include_top=False, weights="imagenet", input_tensor=input_layer)
effnet.trainable = False  # freeze semua layer awal

x = GlobalAveragePooling2D()(effnet.output)
x = Dropout(0.7)(x)
output = Dense(train_gen.num_classes, activation="softmax")(x)

efficient_model = Model(inputs=input_layer, outputs=output)

# ====== Compile Stage 1 (Feature Extractor) ======
efficient_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top_2_acc")]
)

callbacks_stage1 = [
    EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    ModelCheckpoint("efficient_stage1.keras", save_best_only=True, monitor="val_loss"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)
]

print("=== Training Stage 1 (Feature Extraction) ===")
history_stage1 = efficient_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=callbacks_stage1,
    verbose=1
)

# ====== Stage 2a: Fine-Tuning Ringan ======
print("=== Training Stage 2a (Fine-Tuning Ringan) ===")
for layer in effnet.layers[-20:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

efficient_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top_2_acc")]
)

callbacks_stage2a = [
    EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    ModelCheckpoint("efficient_stage2a.keras", save_best_only=True, monitor="val_loss"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)
]

history_stage2a = efficient_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    callbacks=callbacks_stage2a,
    verbose=1
)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

# ====== Evaluasi ROC AUC ======
# Prediksi probabilitas di data test
y_pred_proba = efficient_model.predict(test_gen)
y_true = test_gen.classes
class_labels = list(test_gen.class_indices.keys())

# Konversi y_true ke one-hot encoding biar sesuai dengan output softmax
from tensorflow.keras.utils import to_categorical
y_true_onehot = to_categorical(y_true, num_classes=len(class_labels))

# Hitung ROC AUC (macro & per kelas)
roc_auc_macro = roc_auc_score(y_true_onehot, y_pred_proba, average="macro")
roc_auc_weighted = roc_auc_score(y_true_onehot, y_pred_proba, average="weighted")

print(f"\nROC AUC (macro average): {roc_auc_macro:.4f}")
print(f"ROC AUC (weighted average): {roc_auc_weighted:.4f}")

# ====== (Opsional) Visualisasi ROC Curve per kelas ======
plt.figure(figsize=(8,6))
for i, label in enumerate(class_labels):
    fpr, tpr, _ = roc_curve(y_true_onehot[:, i], y_pred_proba[:, i])
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{label} (AUC = {auc_score:.3f})")

plt.plot([0,1], [0,1], "k--")  # garis diagonal
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve per Class (EfficientNetB1 Model)")
plt.legend(loc="lower right")
plt.show()


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# ====== Setup folder dataset ======
base_dir = "/content/drive/MyDrive/FNNPK_SPLIT"
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# ====== Data Generator ======
train_datagen = ImageDataGenerator(
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
)

val_test_datagen = ImageDataGenerator()

train_gen = train_datagen.flow_from_directory(
    os.path.join(base_dir, "train"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_gen = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, "val"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_gen = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, "test"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

# ====== Model DenseNet121 ======
input_layer = Input(shape=(224, 224, 3))
densenet = DenseNet121(include_top=False, weights="imagenet", input_tensor=input_layer)
densenet.trainable = False

x = GlobalAveragePooling2D()(densenet.output)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(train_gen.num_classes, activation="softmax")(x)

dense_model = Model(inputs=input_layer, outputs=output)

# ====== Compile Stage 1 (Feature Extractor) ======
dense_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy", "categorical_crossentropy", tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top_2_acc")]
)

callbacks_stage1 = [
    EarlyStopping(monitor="val_categorical_crossentropy", patience=5, restore_best_weights=True),
    ModelCheckpoint("densenet_densehead_stage1.keras", save_best_only=True, monitor="val_categorical_crossentropy"),
    ReduceLROnPlateau(monitor="val_categorical_crossentropy", factor=0.5, patience=3, min_lr=2e-7)
]

print("=== Training Stage 1 (Feature Extraction) ===")
history_stage1 = dense_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=callbacks_stage1,
    verbose=1
)

# ====== Stage 2: Fine-Tuning ======
print("=== Training Stage 2 (Fine-Tuning) ===")

# Membuka 50 layer terakhir untuk fine-tuning
for layer in densenet.layers[-50:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

dense_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy", "categorical_crossentropy", tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top_2_acc")]
)

callbacks_stage2 = [
    EarlyStopping(monitor="val_categorical_crossentropy", patience=5, restore_best_weights=True),
    ModelCheckpoint("densenet_densehead_stage2.keras", save_best_only=True, monitor="val_categorical_crossentropy"),
    ReduceLROnPlateau(monitor="val_categorical_crossentropy", factor=0.5, patience=3, min_lr=1e-7)
]

print("=== Starting Fine-Tuning ===")
history_stage2 = dense_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    callbacks=callbacks_stage2,
    verbose=1
)

print("=== Pelatihan Selesai ===")

In [ ]:
# ====== Evaluasi ROC AUC untuk DenseNet121 ======
from sklearn.metrics import roc_auc_score, roc_curve, auc
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import numpy as np

print("\n=== Evaluasi ROC AUC (Multi-class) ===")

# Prediksi probabilitas di data test
y_pred_proba = dense_model.predict(test_gen)
y_true = test_gen.classes
class_labels = list(test_gen.class_indices.keys())

# Ubah y_true ke one-hot encoding
y_true_onehot = to_categorical(y_true, num_classes=len(class_labels))

# Hitung nilai AUC
roc_auc_macro = roc_auc_score(y_true_onehot, y_pred_proba, average="macro")
roc_auc_weighted = roc_auc_score(y_true_onehot, y_pred_proba, average="weighted")

print(f"ROC AUC (macro average): {roc_auc_macro:.4f}")
print(f"ROC AUC (weighted average): {roc_auc_weighted:.4f}")

# ====== (Opsional) Visualisasi ROC Curve per kelas ======
plt.figure(figsize=(8,6))
for i, label in enumerate(class_labels):
    fpr, tpr, _ = roc_curve(y_true_onehot[:, i], y_pred_proba[:, i])
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{label} (AUC = {auc_score:.3f})")

plt.plot([0,1], [0,1], "k--")  # garis diagonal acak
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve per Class (DenseNet121 Model)")
plt.legend(loc="lower right")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB1, DenseNet121
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# ====== Input ======
IMG_SIZE = (224,224,3)
num_classes = train_gen.num_classes

input_layer = tf.keras.Input(shape=IMG_SIZE)

# ====== EfficientNetB1 Backbone ======
effnet = EfficientNetB1(include_top=False, weights="imagenet", input_tensor=input_layer)
effnet.trainable = False
x_eff = layers.GlobalAveragePooling2D()(effnet.output)

# ====== DenseNet121 Backbone ======
densenet = DenseNet121(include_top=False, weights="imagenet", input_tensor=input_layer)
densenet.trainable = False
x_dense = layers.GlobalAveragePooling2D()(densenet.output)

# ====== Gabungkan feature ======
x = layers.concatenate([x_eff, x_dense])
x = layers.Dropout(0.5)(x)
output = layers.Dense(num_classes, activation="softmax")(x)

# ====== Hybrid Model ======
hybrid_model = Model(inputs=input_layer, outputs=output)

# ====== Compile tahap 1 (freeze backbone) ======
hybrid_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top_2_acc")]
)

callbacks_stage1 = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint("hybrid_stage1.keras", save_best_only=True, monitor="val_loss", verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# ====== Training tahap 1 ======
history_stage1 = hybrid_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=callbacks_stage1,
    verbose=1
)

# ====== Fine-tuning backbone ======
trainable_layers_eff = 20
trainable_layers_dense = 20

for layer in effnet.layers[-trainable_layers_eff:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

for layer in densenet.layers[-trainable_layers_dense:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

# Compile ulang dengan learning rate lebih kecil
hybrid_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top_2_acc")]
)

callbacks_stage2 = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint("hybrid_stage2.keras", save_best_only=True, monitor="val_loss", verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

# ====== Training tahap 2 ======
history_stage2 = hybrid_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    callbacks=callbacks_stage2,
    verbose=1
)

In [ ]:
# ====== Evaluasi ROC AUC untuk Hybrid Model (EfficientNetB1 + DenseNet121) ======
from sklearn.metrics import roc_auc_score, roc_curve, auc
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import numpy as np

print("\n=== Evaluasi ROC AUC (Hybrid Model) ===")

# Prediksi probabilitas di data test
y_pred_proba = hybrid_model.predict(test_gen)
y_true = test_gen.classes
class_labels = list(test_gen.class_indices.keys())

# Ubah y_true ke one-hot encoding
y_true_onehot = to_categorical(y_true, num_classes=len(class_labels))

# Hitung nilai ROC AUC
roc_auc_macro = roc_auc_score(y_true_onehot, y_pred_proba, average="macro")
roc_auc_weighted = roc_auc_score(y_true_onehot, y_pred_proba, average="weighted")

print(f"ROC AUC (macro average): {roc_auc_macro:.4f}")
print(f"ROC AUC (weighted average): {roc_auc_weighted:.4f}")

# ====== (Opsional) Visualisasi ROC Curve per kelas ======
plt.figure(figsize=(8,6))
for i, label in enumerate(class_labels):
    fpr, tpr, _ = roc_curve(y_true_onehot[:, i], y_pred_proba[:, i])
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{label} (AUC = {auc_score:.3f})")

plt.plot([0,1], [0,1], "k--")  # garis diagonal
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve per Class (Hybrid Model)")
plt.legend(loc="lower right")
